# Oxygen sampling–mapping trend differences

Reproduces the two-period figure with Olivelli et al. (2026). Column 1 covers 1965–2021 and column 2 covers 1993–2021. The notebook saves both the original overlapping-distribution design and a box-and-whisker design to .


In [1]:
"""Plot global oxygen sampling/mapping trend differences for two periods.

Column 1 is the full 1965--2021 record and column 2 retains 1993--2021.
Two PDF variants are written: the original overlaid half-violin design and a
horizontal box-and-whisker design showing every product plus the combined set.
"""

import os
import pickle
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-db9274")

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter
import numpy as np


SOURCE_DIR = Path("/scratch/gpfs/LRGROUP/db9274/Variability_trends_EC/Paper_figures")
CACHE_FILE = SOURCE_DIR / (
    "cache_sampling_mapping_trends_different_periods/"
    "global_trend_sampling_mapping_samples_full_upper_below_1965_2021_"
    "6_products_olivelli_v3.pkl"
)
OUTPUT_DIR = Path.cwd() / "Figures"
VIOLIN_OUTPUT = OUTPUT_DIR / "oxygen_sampling_mapping_difference_trends_two_periods.pdf"
BOX_OUTPUT = OUTPUT_DIR / "oxygen_sampling_mapping_difference_trends_two_periods_boxplots.pdf"

PERIODS = ["1965–2021", "1993–2021"]
DEPTHS = ["Full water column", "Upper 2000 m", "Below 2000 m"]
PRODUCTS = [
    "observation-Gouretski(2024)",
    "observation-Ito(2024; NN)",
    "observation-Ito(2024; RF)",
    "observation-Ito(2022; 5 year)",
    "observation-Roach and Bindoff",
    "observation-Olivelli(2026)",
]
LABELS = {
    "observation-Gouretski(2024)": "Gouretski et al. (2024)",
    "observation-Ito(2024; NN)": "Ito et al. (2024): Neural network",
    "observation-Ito(2024; RF)": "Ito et al. (2024): Random forest",
    "observation-Ito(2022; 5 year)": "Ito (2022)",
    "observation-Roach and Bindoff": "Roach & Bindoff (2023)",
    "observation-Olivelli(2026)": "Olivelli et al. (2026)",
}
COLORS = dict(zip(PRODUCTS, ["blue", "green", "red", "purple", "#D55E00", "#7B2CBF"]))
XLABEL = "Sampling–mapping effect ($\\mu$mol kg$^{-1}$ decade$^{-1}$)"

plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "figure.dpi": 130,
    "savefig.dpi": 300,
})


def gaussian_pdf(values, grid):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    std = np.std(values, ddof=1)
    bandwidth = 1.06 * std * values.size ** (-1 / 5)
    if not np.isfinite(bandwidth) or bandwidth == 0:
        bandwidth = max(np.ptp(values), 1.0) / 20
    scaled = (grid[:, None] - values[None, :]) / bandwidth
    return np.exp(-0.5 * scaled**2).sum(axis=1) / (
        values.size * bandwidth * np.sqrt(2 * np.pi)
    )


def symmetric_xlim(samples):
    selected = samples[samples["period"].isin(PERIODS)]
    values = selected["sampling_mapping_difference"].to_numpy(float)
    values = values[np.isfinite(values)]
    lo, hi = np.percentile(values, [1, 99])
    extent = 1.2 * max(abs(lo), abs(hi))
    return -extent, extent


def add_percent_axis(ax, data):
    full = data.drop_duplicates(["model", "ensemble"])["fully_sampled_trend"].to_numpy(float)
    reference = abs(np.nanmedian(full))
    if np.isfinite(reference) and reference > 0:
        sec = ax.secondary_xaxis(
            "top",
            functions=(lambda x: 100 * x / reference, lambda pct: pct * reference / 100),
        )
        sec.xaxis.set_major_formatter(FuncFormatter(lambda value, pos: f"{value:g}%"))
        sec.tick_params(axis="x", labelsize=8, pad=1.5)


def panel_data(samples, period, depth):
    return samples[(samples["period"] == period) & (samples["depth"] == depth)]


def plot_half_violin(ax, grid, values, side, color, alpha, linewidth):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    density = gaussian_pdf(values, grid)
    width = side * 0.38 * density / np.nanmax(density)
    ax.fill_between(grid, 0, width, color=color, alpha=alpha, linewidth=0)
    ax.plot(grid, width, color=color, lw=linewidth)
    median = np.median(values)
    ax.plot([median, median], [0, side * 0.35], color=color, lw=1.5)


def format_common_panel(ax, xlim, data, bottom):
    ax.axvline(0, color="0.35", lw=0.9, ls="--")
    ax.set_xlim(xlim)
    ax.grid(axis="x", ls=":", alpha=0.25)
    ax.tick_params(axis="both", labelsize=8.5, pad=1.5)
    ax.set_xlabel(XLABEL if bottom else "")
    add_percent_axis(ax, data)


def make_violin_figure(samples, xlim):
    fig, axes = plt.subplots(3, 2, figsize=(8, 5.5), squeeze=False)
    fig.subplots_adjust(left=0.12, right=0.98, top=0.91, bottom=0.17, wspace=0.16, hspace=0.42)
    grid = np.linspace(*xlim, 300)
    panel = 0
    for row, depth in enumerate(DEPTHS):
        for col, period in enumerate(PERIODS):
            ax = axes[row, col]
            data = panel_data(samples, period, depth)
            plot_half_violin(ax, grid, data["sampling_mapping_difference"], 1, "black", 0.18, 1.9)
            for product in PRODUCTS:
                values = data.loc[data["product"] == product, "sampling_mapping_difference"]
                plot_half_violin(ax, grid, values, -1, COLORS[product], 0.24, 1.7)
            ax.axhline(0, color="0.25", lw=1)
            ax.set_ylim(-0.5, 0.5)
            ax.set_yticks([])
            format_common_panel(ax, xlim, data, row == 2)
            if row == 0:
                ax.set_title(period, pad=24)
            if col == 0:
                ax.set_ylabel(depth, fontsize=9.5, fontweight="bold", labelpad=22)
            ax.text(0.01, 0.98, chr(ord("a") + panel), transform=ax.transAxes,
                    ha="left", va="top", fontsize=11, fontweight="bold")
            panel += 1
    handles = [Patch(facecolor="black", alpha=0.18, label="Combined estimate")]
    handles += [Patch(facecolor=COLORS[p], alpha=0.35, label=LABELS[p]) for p in PRODUCTS]
    fig.legend(handles=handles, loc="lower center", ncol=4, frameon=False, fontsize=7.7,
               bbox_to_anchor=(0.55, -0.005), columnspacing=0.8, handlelength=1.1)
    fig.savefig(VIOLIN_OUTPUT, bbox_inches="tight")
    plt.close(fig)


def make_box_figure(samples, xlim):
    fig, axes = plt.subplots(3, 2, figsize=(10.5, 9.0), squeeze=False)
    fig.subplots_adjust(left=0.12, right=0.98, top=0.94, bottom=0.15, wspace=0.16, hspace=0.32)
    box_labels = ["Combined estimate"] + [LABELS[p] for p in PRODUCTS]
    box_colors = ["black"] + [COLORS[p] for p in PRODUCTS]
    positions = np.arange(len(box_labels), 0, -1)
    panel = 0
    for row, depth in enumerate(DEPTHS):
        for col, period in enumerate(PERIODS):
            ax = axes[row, col]
            data = panel_data(samples, period, depth)
            groups = [data["sampling_mapping_difference"].dropna().to_numpy()]
            groups += [data.loc[data["product"] == p, "sampling_mapping_difference"].dropna().to_numpy()
                       for p in PRODUCTS]
            result = ax.boxplot(groups, vert=False, positions=positions, widths=0.58,
                                patch_artist=True, showfliers=True,
                                medianprops={"color": "white", "linewidth": 1.4},
                                whiskerprops={"linewidth": 1}, capprops={"linewidth": 1},
                                flierprops={"marker": "o", "markersize": 2.8, "alpha": 0.55})
            for patch, color in zip(result["boxes"], box_colors):
                patch.set(facecolor=color, edgecolor=color, alpha=0.5)
            ax.set_yticks(positions)
            ax.set_yticklabels([])
            ax.set_ylim(0.35, len(box_labels) + 0.65)
            format_common_panel(ax, xlim, data, row == 2)
            if row == 0:
                ax.set_title(period, pad=24)
            if col == 0:
                ax.text(-0.14, 0.5, depth, transform=ax.transAxes, rotation=90,
                        va="center", ha="center", fontsize=9.5, fontweight="bold")
            ax.text(0.01, 0.98, chr(ord("a") + panel), transform=ax.transAxes,
                    ha="left", va="top", fontsize=11, fontweight="bold")
            panel += 1
    handles = [Patch(facecolor="black", edgecolor="black", alpha=0.5, label="Combined estimate")]
    handles += [Patch(facecolor=COLORS[p], edgecolor=COLORS[p], alpha=0.5, label=LABELS[p])
                for p in PRODUCTS]
    fig.legend(handles=handles, loc="lower center", ncol=4, frameon=False, fontsize=8.0,
               bbox_to_anchor=(0.55, 0.005), columnspacing=0.9, handlelength=1.5)
    fig.savefig(BOX_OUTPUT, bbox_inches="tight")
    plt.close(fig)


def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    with CACHE_FILE.open("rb") as handle:
        samples = pickle.load(handle)
    missing_products = set(PRODUCTS) - set(samples["product"])
    missing_periods = set(PERIODS) - set(samples["period"])
    if missing_products or missing_periods:
        raise ValueError(f"Cache missing products={missing_products}, periods={missing_periods}")
    xlim = symmetric_xlim(samples)
    make_violin_figure(samples, xlim)
    make_box_figure(samples, xlim)
    print(f"Saved {VIOLIN_OUTPUT}")
    print(f"Saved {BOX_OUTPUT}")


if __name__ == "__main__":
    main()


Saved /scratch/gpfs/LRGROUP/db9274/Variability_trends_EC/Paper_figures_p2/Figures/oxygen_sampling_mapping_difference_trends_two_periods.pdf
Saved /scratch/gpfs/LRGROUP/db9274/Variability_trends_EC/Paper_figures_p2/Figures/oxygen_sampling_mapping_difference_trends_two_periods_boxplots.pdf


In [2]:

# Requested boxplot + Ito (2024) RF sampling–mapping maps (1965–2021 only).
import xarray as xr
from matplotlib.colors import TwoSlopeNorm, ListedColormap
from cartopy import crs as ccrs
import cartopy.feature as cfeature
MAP_CENTRAL_LONGITUDE = 205
SEAM_INTERPOLATION_LONGITUDES = (71.5, 72.5, 73.5, 74.5)
MAX_SEAM_INTERPOLATION_GAP_DEGREES = 80.0

MAP_FILE = Path('/scratch/gpfs/LRGROUP/db9274/Subsampled_o2_variability/Cause_model_obs_differences/Figures/all_models_multimodel_mean_mapped_minus_full_deoxygenation_difference_maps_1965_2021.nc')
FULL_FILE = SOURCE_DIR / 'NETCDF/Fully_sampled_models_annual_o2/o2_1x1bin_GFDL_ESM4_r1i1p1f1.nc'

def _layer_edges(z):
    e=np.empty(len(z)+1); e[1:-1]=(z[:-1]+z[1:])/2; e[0]=max(0,z[0]-(z[1]-z[0])/2); e[-1]=z[-1]+(z[-1]-z[-2])/2; return e

def rf_maps():
    with xr.open_dataset(MAP_FILE) as ds:
        i=int(np.flatnonzero(ds.product.values == 'Ito et al. (2024): RF')[0])
        upper=ds.difference.isel(product=i, depth=0).values
        lower=ds.difference.isel(product=i, depth=1).values
        lat,lon=ds.y.values,ds.x.values
        lon=((lon + 180) % 360) - 180
        order=np.argsort(lon); lon=lon[order]
        upper=upper[:, order]; lower=lower[:, order]
    with xr.open_dataset(FULL_FILE) as ds:
        z=ds['lev'].values; a=ds['o2'].isel(time=0).values
    e=_layer_edges(z); u=np.sum(np.where(np.isfinite(a),np.maximum(np.minimum(e[1:],2000)-e[:-1],0)[:,None,None],0),axis=0); l=np.sum(np.where(np.isfinite(a),np.maximum(e[1:]-np.maximum(e[:-1],2000),0)[:,None,None],0),axis=0)
    den=u+l; full=np.divide(upper*u+lower*l,den,out=np.full_like(upper,np.nan),where=den>0)
    return lat,lon,[full,upper,lower]
def _prepare_longitude_for_plot(field, lon):
    # Match the periodic native-column interpolation used by the source map figure.
    result=np.array(field,copy=True); x=np.asarray(lon,dtype=float); dx=float(np.nanmedian(np.diff(x)))
    targets=[]
    for target in SEAM_INTERPOLATION_LONGITUDES:
        while target < float(x.min()): target += 360.0
        while target > float(x.max()) + dx: target -= 360.0
        targets.append(target)
    native=sorted(float(x[np.argmin(np.abs(x-target))]) for target in targets if np.any(np.isclose(x,target)))
    groups=[]
    for target in native:
        if not groups or not np.isclose(target-groups[-1][-1],dx): groups.append([target])
        else: groups[-1].append(target)
    for group in groups:
        indices=[int(np.argmin(np.abs(x-target))) for target in group]; start=min(indices); end=max(indices); nx=len(x)
        for row in result:
            search=row.copy(); search[indices]=np.nan; left=right=None
            for step in range(1,nx):
                idx=(start-step)%nx
                if np.isfinite(search[idx]): left=idx; break
            for step in range(1,nx):
                idx=(end+step)%nx
                if np.isfinite(search[idx]): right=idx; break
            if left is None or right is None: continue
            left_x=float(x[left]); right_x=float(x[right])+(360.0 if right<=left else 0.0); span=right_x-left_x
            if span<=0 or span>MAX_SEAM_INTERPOLATION_GAP_DEGREES: continue
            for idx in indices:
                if np.isfinite(row[idx]): continue
                target=float(x[idx])+(360.0 if idx<=left else 0.0); weight=(target-left_x)/span
                row[idx]=row[left]+weight*(row[right]-row[left])
    return np.concatenate([result,result[:,:1]],axis=1),np.r_[x,x[-1]+dx]


def make_requested_box_map_figure(samples,xlim):
    fig=plt.figure(figsize=(9.5,8))
    gs=fig.add_gridspec(3,2,left=.12,right=.98,top=.94,bottom=.20,wspace=.16,hspace=.34)
    labels=['Combined estimate']+[LABELS[p] for p in PRODUCTS]; colors=['black']+[COLORS[p] for p in PRODUCTS]; pos=np.arange(len(labels),0,-1)
    for row,depth in enumerate(DEPTHS):
        ax=fig.add_subplot(gs[row,0]); data=panel_data(samples,'1965–2021',depth); groups=[data.sampling_mapping_difference.dropna().to_numpy()]+[data.loc[data["product"]==p,'sampling_mapping_difference'].dropna().to_numpy() for p in PRODUCTS]
        r=ax.boxplot(groups,vert=False,positions=pos,widths=.58,patch_artist=True,showfliers=True,medianprops={'color':'white','linewidth':1.4},flierprops={'marker':'o','markersize':2.8,'alpha':.55})
        for patch,color in zip(r['boxes'],colors): patch.set(facecolor=color,edgecolor=color,alpha=.5)
        ax.set_yticks(pos); ax.set_yticklabels([]); ax.set_ylim(.35,len(labels)+.65); format_common_panel(ax,xlim,data,row==2); ax.set_title(f'{chr(97+2*row)}. {depth}',fontweight='bold')
    lat,lon,fields=rf_maps(); finite=np.concatenate([f[np.isfinite(f)] for f in fields]); lim=float(np.nanpercentile(np.abs(finite),99)); norm=TwoSlopeNorm(vmin=-lim,vcenter=0,vmax=lim); mesh=None
    for row,(depth,field) in enumerate(zip(DEPTHS,fields)):
        ax=fig.add_subplot(gs[row,1],projection=ccrs.Robinson(central_longitude=MAP_CENTRAL_LONGITUDE))
        cyclic_field,cyclic_lon=_prepare_longitude_for_plot(field,lon)
        invalid=np.where(np.isfinite(cyclic_field),np.nan,1.0)
        ax.pcolormesh(cyclic_lon,lat,invalid,transform=ccrs.PlateCarree(),cmap=ListedColormap(['.86']),vmin=0,vmax=1,shading='auto',rasterized=True,zorder=1)
        mesh=ax.pcolormesh(cyclic_lon,lat,cyclic_field,transform=ccrs.PlateCarree(),cmap='bwr',norm=norm,shading='auto',rasterized=True,zorder=2)
        ax.set_global(); ax.add_feature(cfeature.LAND,facecolor='.86',edgecolor='none',zorder=3); ax.coastlines(linewidth=.45,color='.15',zorder=4); ax.gridlines(linewidth=.25,color='.55',alpha=.4); ax.spines['geo'].set_edgecolor('.15'); ax.spines['geo'].set_linewidth(.6); ax.set_title(f'{chr(98+2*row)}. {depth} Ito et al. (2024): RF',fontweight='bold')
    handles=[Patch(facecolor='black',edgecolor='black',alpha=.5,label='Combined estimate')]+[Patch(facecolor=COLORS[p],edgecolor=COLORS[p],alpha=.5,label=LABELS[p]) for p in PRODUCTS]
    fig.legend(handles=handles,loc='lower center',bbox_to_anchor=(.32,.035),ncol=2,frameon=False,fontsize=8)
    cax=fig.add_axes([.63,.15,.29,.022]); cb=fig.colorbar(mesh,cax=cax,orientation='horizontal',extend='both'); cb.set_label('Sampled-and-mapped − fully sampled\ndeoxygenation difference (µmol kg⁻¹ decade⁻¹)')
    fig.savefig(BOX_OUTPUT,bbox_inches='tight'); plt.close(fig); Path(VIOLIN_OUTPUT).unlink(missing_ok=True)

with CACHE_FILE.open('rb') as h: _samples=pickle.load(h)
make_requested_box_map_figure(_samples,symmetric_xlim(_samples))
print(f'Saved {BOX_OUTPUT}')


Saved /scratch/gpfs/LRGROUP/db9274/Variability_trends_EC/Paper_figures_p2/Figures/oxygen_sampling_mapping_difference_trends_two_periods_boxplots.pdf


In [3]:

# Supplementary figure: boxplots for 1993–2021 only.
SUPP_OUTPUT = OUTPUT_DIR / 'oxygen_sampling_mapping_difference_trends_boxplots_1993_2021_supplement.pdf'

def make_supplementary_boxplots(samples, xlim):
    fig, axes = plt.subplots(1, 3, figsize=(12.5, 4.6), sharex=True)
    fig.subplots_adjust(left=0.08, right=0.98, top=0.94, bottom=0.27, wspace=0.18)
    labels = ['Combined estimate'] + [LABELS[p] for p in PRODUCTS]
    colors = ['black'] + [COLORS[p] for p in PRODUCTS]
    positions = np.arange(len(labels), 0, -1)
    for ax, depth, letter in zip(axes, DEPTHS, ('a', 'b', 'c')):
        data = panel_data(samples, '1993–2021', depth)
        groups = [data['sampling_mapping_difference'].dropna().to_numpy()]
        groups += [data.loc[data['product'] == p, 'sampling_mapping_difference'].dropna().to_numpy() for p in PRODUCTS]
        result = ax.boxplot(groups, vert=False, positions=positions, widths=0.58,
                            patch_artist=True, showfliers=True,
                            medianprops={'color': 'white', 'linewidth': 1.4},
                            whiskerprops={'linewidth': 1}, capprops={'linewidth': 1},
                            flierprops={'marker': 'o', 'markersize': 2.8, 'alpha': 0.55})
        for patch, color in zip(result['boxes'], colors):
            patch.set(facecolor=color, edgecolor=color, alpha=0.5)
        ax.axvline(0, color='0.35', lw=0.9, ls='--')
        ax.set_xlim(xlim); ax.set_ylim(0.35, len(labels) + 0.65)
        ax.set_yticks(positions); ax.set_yticklabels([])
        ax.grid(axis='x', ls=':', alpha=0.25)
        ax.tick_params(axis='both', labelsize=8.5, pad=1.5)
        ax.set_xlabel(XLABEL, fontsize=9)
        ax.set_title(f'{letter}. {depth}', fontweight='bold')
        add_percent_axis(ax, data)
    handles = [Patch(facecolor='black', edgecolor='black', alpha=0.5, label='Combined estimate')]
    handles += [Patch(facecolor=COLORS[p], edgecolor=COLORS[p], alpha=0.5, label=LABELS[p]) for p in PRODUCTS]
    fig.legend(handles=handles, loc='lower left',
               bbox_to_anchor=(0.06, 0.02, 0.92, 0.13), mode='expand',
               ncols=4, frameon=False, fontsize=8.0, borderaxespad=0,
               columnspacing=1.0, handlelength=1.5)
    fig.savefig(SUPP_OUTPUT, bbox_inches='tight')
    plt.close(fig)

with CACHE_FILE.open('rb') as handle:
    _supp_samples = pickle.load(handle)
make_supplementary_boxplots(_supp_samples, symmetric_xlim(_supp_samples))
print(f'Saved {SUPP_OUTPUT}')


Saved /scratch/gpfs/LRGROUP/db9274/Variability_trends_EC/Paper_figures_p2/Figures/oxygen_sampling_mapping_difference_trends_boxplots_1993_2021_supplement.pdf
